> **Edición pública anonimizada.** Nombres y rutas comerciales se sustituyeron por etiquetas genéricas; no se incluyen bases ni modelos originales. Los resultados históricos se consultan en la tesis adaptada. La demostración pública usa datos y modelos sintéticos.


In [ ]:
from pathlib import Path
import os, sys

# Find the repository independently of the computer or clone location.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / ".tesis-portafolio").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Abre Jupyter desde la carpeta de este repositorio.")
if (REPO_ROOT / "research").is_dir():
    REPO_ROOT = REPO_ROOT / "research"
os.chdir(REPO_ROOT / '05 Power BI')


# Documentación de componentes — carpeta `05 Power BI`

## Qué es este notebook

La carpeta `05 Power BI` es el último eslabón del proyecto: convierte los dos
modelos finales del proyecto —la segmentación K-Means y el modelo predictivo de
conversión— en **un paso automático dentro de Power BI**. Cuando el analista
actualiza el informe, cada lead recibe dos columnas nuevas: `cluster` (a qué
perfil de cliente se parece) y `percentil` (qué tan prioritario es contactarlo).

Ese resultado no lo produce un solo archivo, sino **siete módulos de Python, tres
artefactos de modelo y una batería de doce pruebas**. Este notebook los recorre
uno por uno, explica qué hace cada cual y —lo importante— **lo ejecuta en vivo**,
de modo que cada afirmación de esta documentación queda respaldada por la salida
que aparece debajo.

> Para *usar* el tablero no hace falta leer este notebook: para eso está el
> [`Paso_2_Guia_Scoring.pdf`](Paso_2_Guia_Scoring.pdf), escrito para una persona sin conocimientos técnicos.
> Este documento es el complemento de auditoría: explica el **cómo** y el
> **por qué** de cada script, y sirve como apéndice técnico de la tesis.

**A quién está dirigido:** se asume que usted sabe Python básico —variables,
funciones, listas, diccionarios, un `for`— pero no se asume que conozca
vocabulario de ciencia de datos. Cada término de ese mundo (»cluster»,
«pipeline», «codificación one-hot»…) se explica en el momento en que aparece
por primera vez, y la siguiente sección los reúne todos en una sola tabla de
consulta.

## Cómo está organizado

Cada componente se documenta con la misma estructura de cuatro apartados que usa
el resto de apéndices del proyecto:

| Apartado | Qué contiene |
|---|---|
| **Propósito** | Qué problema resuelve ese archivo y por qué existe como pieza aparte |
| **Entradas** | Qué recibe: datos, artefactos u otros módulos |
| **Qué hace** | La lógica, explicada en prosa; las decisiones de diseño y sus motivos |
| **Salidas** | Qué devuelve y quién lo consume después |

Después de cada explicación viene una celda de código que **demuestra ese
componente aislado**, sobre una tabla de ejemplo que imita una exportación real
del CRM con casos difíciles puestos a propósito.

## Cómo ejecutarlo

**Kernel → Restart Kernel and Run All Cells**, desde la carpeta `05 Power BI`.
No modifica ningún archivo del proyecto: solo lee, calcula y muestra. 

---

## Vocabulario para leer este notebook

Antes de entrar en materia, una tabla de referencia para los términos que van a
aparecer una y otra vez. No hace falta memorizarla: vuelva aquí cuando alguna
palabra no le haga sentido en el resto del documento.

| Término | Qué significa |
|---|---|
| **DataFrame** | La tabla de la biblioteca `pandas`: filas y columnas, como una hoja de cálculo pero dentro de Python. En este notebook, casi siempre es la tabla de leads. |
| **Series** | Una sola columna de un DataFrame, considerada aparte. |
| **Nombre / variable canónica** | El nombre **fijo** que este proyecto usa por dentro para referirse a una variable, sin importar cómo venga escrita en la tabla de origen. Ejemplo: el CRM puede llamarla `estado_civil_label` y la limpieza la llamó `Estado Cívil`, pero dentro del código siempre se le dice **`estado_civil`** — ese es su nombre canónico. «Canónico» aquí solo quiere decir «el nombre oficial que usamos nosotros». |
| **Alias** | Cualquier otro nombre con el que esa misma variable puede llegar (`Estado Cívil`, `estado_civil_label`…). El código los reconoce todos y los traduce al nombre canónico. |
| **Limpiar / normalizar un valor** | Corregir cómo está escrito un dato sin cambiar lo que significa: quitar espacios de más, unificar mayúsculas, unir `'casado'` y `'CASADA'` en una sola forma escrita. |
| **Variable derivada** | Una columna que no viene del CRM tal cual, sino que se calcula a partir de otras columnas. Ejemplo: `mayor_45` no existe en el CRM — se calcula como `edad >= 45`. |
| **Modelo entrenado** | Un programa que «aprendió» un patrón a partir de datos históricos (los leads de la tesis) y ahora puede aplicar ese patrón a leads nuevos que nunca vio. Queda guardado en un archivo (`.pkl`, `.joblib`) que no se edita a mano — es un archivo de datos, no de código. |
| **K-Means / cluster / centroide** | K-Means es el algoritmo que agrupó a los leads en 6 perfiles distintos, llamados **clusters**. Cada cluster tiene un **centroide**: un punto que representa el "promedio" de ese grupo. A un lead nuevo se le asigna el cluster cuyo centroide le queda más cerca. |
| **Codificación one-hot** (*one-hot encoding*) | La forma en que un modelo "lee" una categoría de texto. Un modelo no entiende la palabra `'Femenino'`; entiende números. Entonces la columna `sexo` se convierte en dos columnas de solo 0 y 1: `sexo_Femenino` (1 si es mujer, 0 si no) y `sexo_Masculino`. Eso es codificar «en one-hot»: una sola columna de la nueva tanda vale 1 («está encendida»), el resto vale 0. |
| **Estandarizar** | Convertir un número (como la edad) a una escala común: restarle el promedio del grupo y dividir entre qué tanto varían normalmente los valores. Sirve para que el modelo no le dé más peso a una variable solo porque sus números son más grandes que los de otra. |
| **Pipeline** | Una cadena de pasos que se aplican uno después del otro, siempre en el mismo orden, guardada como un solo objeto de Python. En este proyecto: rellenar los datos faltantes → convertir el texto a columnas de 0 y 1 → poner los números en una escala común → aplicar el modelo. Guardar todo como un pipeline evita tener que acordarse de repetir esos pasos a mano cada vez que se puntúa un lead nuevo. |
| **Imputar** (valores faltantes) | Rellenar un dato que está vacío con un valor razonable, en vez de dejarlo en blanco. Por ejemplo: la mediana de la edad de los demás leads, o la palabra `'_missing_'` en una columna de texto. |
| **Regresión logística** | El tipo de modelo que se usa para calcular la probabilidad de conversión. Toma los datos de un lead y devuelve un número entre 0 y 1: qué tan probable es, según lo que el modelo aprendió, que ese lead termine comprando. |
| **Probabilidad vs. percentil** | La **probabilidad** es ese número entre 0 y 1 que calcula el modelo para un lead en particular. El **percentil** es la posición de ese lead frente a todos los demás leads del lote, expresada de 0 a 100. Este proyecto publica el percentil, no la probabilidad — la sección 9 explica por qué. |
| **Ranking porcentual** | La operación que convierte probabilidades en percentiles: ordena a todos los leads de menor a mayor probabilidad, y a cada uno le asigna qué porcentaje de leads quedó por debajo de él. |
| **Caché** (`lru_cache`) | Guardar en memoria algo que ya se calculó o se leyó una vez, para no tener que repetir el trabajo. Aquí se usa para no volver a leer los archivos de modelo desde el disco cada vez que hacen falta dentro de una misma ejecución. |
| **Huella / hash MD5** | Una especie de huella digital de un archivo: un código corto que se calcula a partir de su contenido exacto. Si el archivo cambia aunque sea en un solo carácter, la huella cambia por completo. Sirve para comprobar que dos archivos son idénticos sin tener que compararlos línea por línea a mano. |
| **Universo (de un modelo)** | El conjunto de leads sobre los que un modelo aprendió, o para los que tiene sentido usarlo. El clustering, por ejemplo, solo aprendió de leads que tenían los cinco datos demográficos completos: ese conjunto es «su universo». Fuera de ese universo, el modelo no tiene forma confiable de opinar. |
| **Excepción / error** | El momento en que el código encuentra algo que no puede procesar correctamente y detiene la ejecución con un mensaje, en vez de seguir adelante y entregar un resultado que sería incorrecto. |

---

## Mapa de la carpeta

```
05 Power BI/
│
├── powerbi_script.py          ← las 8 líneas que se pegan en Power BI
├── requirements.txt           ← versiones exactas de las dependencias
├── pyproject.toml             ← define el paquete instalable
├── Paso_2_Guia_Scoring.pdf                  ← guía de uso, sin tecnicismos
│
├── scoring_leads/             ← el paquete: todo el cálculo vive aquí
│   ├── __init__.py            ← orquestador y guardián del contrato
│   ├── columnas.py            ← ¿cómo se llama cada columna en esta tabla?
│   ├── normalizacion.py       ← limpieza de valores (copia literal del pipeline)
│   ├── preparacion.py         ← construye las variables que piden los modelos
│   ├── artefactos.py          ← abre los archivos de modelo
│   ├── clustering.py          ← calcula `cluster`
│   ├── prediccion.py          ← calcula `percentil`
│   └── modelos/               ← los tres artefactos entrenados
│
└── pruebas/
    └── test_contrato_dataset.py   ← 12 pruebas del contrato con Power BI
```

## Cómo se encadenan

Los módulos no están sueltos: forman una tubería con una dirección única. La
tabla entra por arriba, y baja convirtiéndose en las dos columnas de salida.

![Flujo del componente de priorización: entrada desde el CRM, preparación, clustering, predicción y salida final](attachment:Grafico.png)

Ambos módulos de modelo (`clustering.py` y `prediccion.py`) piden sus artefactos
a `artefactos.py`, que los carga una sola vez y los mantiene en memoria.

**Una decisión de diseño que explica toda la estructura:** cada módulo tiene una
sola responsabilidad y ninguno depende del que va después. `columnas.py` no sabe
qué es un modelo; `normalizacion.py` no sabe de dónde salieron los valores;
`clustering.py` no sabe que existe Power BI. Así, cuando el CRM cambia un nombre
de columna se toca un único archivo, y cuando se reentrena un modelo se sustituye
un único artefacto.

---

# 0. Configuración y verificación del entorno

Antes de documentar nada, se comprueba que el entorno esté completo: que el
paquete se pueda importar, que las versiones de las bibliotecas sean las
correctas y que los tres artefactos de modelo estén en su sitio.

La versión de `scikit-learn` merece atención especial. Los modelos se guardaron
con la **1.9.0**, y un archivo `.pkl` o `.joblib` no es un formato neutro: es una
fotografía de objetos de esa versión concreta. Abrirlo con otra versión mayor
puede fallar, o —el caso peligroso— funcionar y devolver resultados distintos sin
advertirlo. Por eso `requirements.txt` la fija con `==` y no con `>=`.

In [ ]:
import sys
from pathlib import Path

CARPETA = Path.cwd()
if not (CARPETA / 'scoring_leads').is_dir():          # por si el kernel arrancó en otro sitio
    CARPETA = Path(__file__).resolve().parent if '__file__' in dir() else CARPETA
sys.path.insert(0, str(CARPETA))

import numpy as np
import pandas as pd
import sklearn

print('Carpeta de trabajo :', CARPETA)
print()
print('Python       ', sys.version.split()[0])
print('pandas       ', pd.__version__)
print('numpy        ', np.__version__)
print('scikit-learn ', sklearn.__version__, '   <-- debe ser 1.9.0')
print()

artefactos = {
    'modelo_kmeans_R1.joblib'         : 'segmentación final K-Means (k = 6)',
    'preprocesamiento_kmeans_R1.json' : 'estadísticos congelados de la matriz R1',
    'modelo_conversion_lr.pkl'        : 'modelo predictivo final (pipeline completo)',
}
print('Artefactos en scoring_leads/modelos/')
for nombre, descripcion in artefactos.items():
    ruta = CARPETA / 'scoring_leads' / 'modelos' / nombre
    estado = f'OK  {ruta.stat().st_size / 1024:8.1f} KB' if ruta.exists() else 'FALTA'
    print(f'  {estado}   {nombre:<34} {descripcion}')

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)

---

# 1. `powerbi_script.py` — el único archivo que ve el usuario

### Propósito

Ser el fragmento que se copia y se pega en Power BI. Todo lo demás vive dentro
del paquete `scoring_leads`; este archivo existe para que la persona que arma el
informe tenga que manipular **ocho líneas y una sola decisión**, no un programa.

### Entradas

* La variable `dataset`, que **Power BI crea automáticamente** con el contenido de
  la consulta antes del paso de Python. No se declara ni se carga: ya está ahí
  cuando el script empieza a ejecutarse.
* `CARPETA_SCORING`: la única línea que el usuario puede necesitar cambiar. Se
  deja vacía si el paquete se instaló con `pip install .`; si no, lleva la ruta de
  la carpeta que contiene `scoring_leads`.

### Qué hace

Añade esa ruta a la lista de lugares donde Python busca módulos, importa la única
función pública del paquete y la aplica a la tabla. Nada más: ni cálculos, ni
configuración, ni reglas de negocio. **Toda la lógica está deliberadamente fuera
de este archivo**, porque es el único que se edita a mano en una máquina que no
tiene control de versiones; cuanto menos contenga, menos puede romperse.

### Salidas

La variable `dataset`, reasignada. Power BI toma esa variable al terminar el
script y la ofrece en la lista de tablas del paso siguiente.

In [ ]:
print((CARPETA / 'powerbi_script.py').read_text(encoding='utf-8'))

---

# 2. La tabla de ejemplo que se usa en todo el notebook

Para demostrar cada componente hace falta una tabla de entrada. Se usa la misma
que emplean las pruebas —`pruebas/test_contrato_dataset.py`— porque **está
construida con casos difíciles a propósito**, no con datos cómodos:

| Fila | Lo que se está poniendo a prueba |
|---|---|
| 101 | Lead completo y limpio: el caso fácil, la referencia |
| 102 | Lead completo, perfil de inversión |
| 103 | Faltan sexo, razón de compra e ingreso |
| 104 | `edad = 150`: valor imposible que la limpieza debe descartar |
| 105 | `sexo = 'F'`, `estado_civil = 'casado'`, `razon = 'INVERSIÓN'`: escrituras que hay que homologar; además un rango de ingreso inventado |
| 106 | Sin edad; `sexo = 'Otro'`; `estado_civil = 'X'` sin sentido |
| 107 | Faltan razón de compra y campaña reconocible |

Además hay columnas que a los modelos no les incumben (`id_trato`, `proyecto`,
`importe_estimado`): están ahí para comprobar que el proceso **no las toca**.

In [ ]:
sys.path.insert(0, str(CARPETA / 'pruebas'))
from test_contrato_dataset import tabla_de_ejemplo

entrada = tabla_de_ejemplo()

print(f'Filas: {len(entrada)}   Columnas: {entrada.shape[1]}')
print()
print(entrada[['id_trato', 'edad', 'sexo', 'estado_civil', 'razon_de_compra',
               'tipo_unidad', 'ingreso_familiar']].to_string(index=False))
print()
print('Columnas que los modelos no usan y deben quedar intactas:')
print('  ', [c for c in entrada.columns if c in ('id_trato', 'proyecto', 'importe_estimado')])

---

# 3. `scoring_leads/__init__.py` — el orquestador y el contrato

### Propósito

Exponer **una sola función pública** —`agregar_cluster_y_percentil()`— y
garantizar que cumpla un compromiso explícito con Power BI. «Función pública»
quiere decir: la única pieza de este paquete que se espera que alguien de
afuera llame directamente. El resto de los archivos (`columnas.py`,
`normalizacion.py`, etc.) son apoyo interno: existen para que esta función
funcione, no para usarse sueltos.

### Entradas

La tabla de leads que entrega Power BI (un `DataFrame`, es decir una tabla de pandas — ver Vocabulario), y dos parámetros opcionales:

| Parámetro | Por defecto | Para qué sirve |
|---|---|---|
| `decimales_percentil` | `2` | Redondeo de la columna `percentil` |
| `asignar_cluster_fuera_de_universo` | `False` | Con `True`, los leads incompletos reciben el centroide más cercano en vez de `-1` |

### Qué hace

Coordina, en este orden: valida que la entrada sea una tabla → comprueba que no
existan ya columnas llamadas `cluster` o `percentil` → **copia** la tabla →
delega la limpieza en `preparar()` → pide el cluster a `clustering.py` y la
probabilidad a `prediccion.py` → añade las dos columnas → y **verifica el
contrato antes de devolver**.

Cuatro decisiones de diseño concentradas aquí:

1. **Copia, nunca muta.** `dataset.copy()` en la línea 73. Power BI puede reusar
   el objeto original; modificarlo en el sitio produciría efectos difíciles de
   rastrear.
2. **Falla temprano y con un mensaje accionable** si la tabla ya trae una columna
   `cluster`. Sobrescribirla en silencio sería peor: el informe mostraría datos
   plausibles pero equivocados.
3. **Asigna por posición, no por índice.** Los resultados llegan como arreglos en
   el orden de las filas; se envuelven en `pd.Series(..., index=resultado.index)`
   para que el índice que traiga Power BI —sea el que sea— no reordene nada.
4. **Se audita a sí misma.** `_validar_contrato()` vuelve a comprobar, sobre el
   resultado ya construido, las cuatro garantías. Si alguna se rompiera, el paso
   falla en Power BI en vez de publicar un tablero con números mal alineados.

Y un caso borde explícito: si la tabla llega **vacía** (0 filas), devuelve las dos
columnas vacías con el tipo correcto sin invocar a los modelos. Power BI evalúa
consultas con cero filas durante la edición, y un modelo que recibe una matriz
vacía lanza una excepción.

### Salidas

La misma tabla más dos columnas al final: `cluster` (entero) y `percentil`
(decimal). **La probabilidad de conversión no se expone nunca**: se calcula, se
usa para ordenar y se descarta. Un porcentaje de compra individual junto al
nombre del cliente invita a lecturas que el modelo no sostiene; la posición
relativa es lo que la operación necesita para priorizar llamadas.

In [ ]:
from scoring_leads import (
    agregar_cluster_y_percentil,
    variables_ausentes,
    CLUSTER_SIN_SEGMENTO,
    COLUMNA_CLUSTER,
    COLUMNA_PERCENTIL,
)

salida = agregar_cluster_y_percentil(entrada)

print(salida[['id_trato', 'edad', 'sexo', 'razon_de_compra', 'cluster', 'percentil']].to_string(index=False))
print()
print('Columnas nuevas :', [c for c in salida.columns if c not in entrada.columns])
print('Filas           :', len(entrada), '->', len(salida))
print('Orden de id_trato intacto :', salida['id_trato'].tolist() == entrada['id_trato'].tolist())
print('Entrada sin mutar         :', entrada.equals(tabla_de_ejemplo()))
print('Probabilidad expuesta     :',
      any('prob' in c.lower() or 'score' in c.lower() for c in salida.columns))

La segunda garantía —fallar antes que sobrescribir— se puede provocar a
propósito: si la consulta ya trae una columna `cluster`, el paso se detiene con
un mensaje que dice exactamente qué hacer en Power Query.

In [ ]:
conflictiva = entrada.copy()
conflictiva['cluster'] = 0

try:
    agregar_cluster_y_percentil(conflictiva)
except ValueError as error:
    print('ValueError levantado, como se espera:')
    print(' ', error)

# El otro caso borde: tabla vacía. No se invoca a los modelos y los tipos se respetan.
vacia = agregar_cluster_y_percentil(entrada.iloc[0:0])
print()
print('Tabla vacía -> filas:', len(vacia))
print('             columnas nuevas:', [c for c in vacia.columns if c not in entrada.columns])
print('             tipos:', vacia['cluster'].dtype, '/', vacia['percentil'].dtype)

---

# 4. `scoring_leads/columnas.py` — reconocer cómo se llama cada variable

### Propósito

Resolver un problema práctico y muy concreto. Piense en la variable «estado
civil». Según en qué momento del proyecto se mire, esa misma información puede
venir en una columna llamada de tres formas distintas:

* `estado_civil` — como se le llama en el análisis final;
* `Estado Cívil` — como quedó nombrada durante la etapa de limpieza;
* `estado_civil_label` — como la exporta el CRM en crudo.

Son la misma variable, con tres disfraces. Este módulo decide, de una vez por
todas, **cómo se le va a llamar por dentro del código**: a esa forma fija se le
dice el **nombre canónico** de la variable (en el Vocabulario de la sección 0
está la definición corta: es simplemente «el nombre oficial que usamos
nosotros»). A las otras dos formas —`Estado Cívil`, `estado_civil_label`— se
les llama sus **alias**: nombres alternativos que también se reconocen.

Sin este módulo, el script solo funcionaría si la tabla trajera exactamente el
nombre `estado_civil`; con él, funciona sin importar cuál de los tres alias
traiga la consulta de Power BI.

### Entradas

La tabla de leads (`DataFrame`) y el nombre canónico de la variable que se
busca, por ejemplo `"estado_civil"`.

### Qué hace

Guarda un diccionario, `ALIAS_COLUMNAS`, que para cada variable canónica lista
todos sus alias posibles, en orden de preferencia:

```python
ALIAS_COLUMNAS["estado_civil"] = ("estado_civil", "Estado Cívil", "Estado Civil", "estado_civil_label")
```

Para encontrar la columna correcta en una tabla dada, compara los nombres sin
fijarse en mayúsculas, acentos ni espacios — con una función auxiliar,
`_clave()`, que reduce cualquier nombre a una forma comparable. Así,
`'Estado Cívil'`, `'estado civil'` y `'ESTADO_CIVIL'` se consideran la misma
columna, porque las tres se convierten en la misma clave interna:
`estadocivil`. Esto responde a un problema real de las exportaciones: los
acentos y los espacios cambian de una versión del CRM a otra, y sin esta
normalización el script dejaría de reconocer la columna por un detalle de
formato.

La decisión importante está en la función `obtener_serie()`: si **ninguno** de
los alias de una variable aparece en la tabla, **no lanza un error** — en su
lugar devuelve una columna llena de valores vacíos (nulos), del mismo largo que
la tabla. La razón: durante el entrenamiento de los modelos, una variable
ausente era un problema que había que resolver antes de continuar; pero en
producción, hacer que el informe completo falle porque el CRM dejó de exportar
`lugar_trabajo` sería desproporcionado. El lead se puntúa igual, con la
información que sí tenga, y las partes del modelo que manejan valores
faltantes (ver Vocabulario: «imputar») se encargan del resto.

La función `variables_ausentes()` hace lo contrario: en vez de rellenar en
silencio, **avisa** qué variables importantes no encontró en la tabla, para que
alguien pueda revisarlo antes de publicar el informe.

### Salidas

Una columna (`Series`) por cada variable que se pida, ya sea la columna real
encontrada bajo cualquiera de sus alias, o una columna de valores vacíos si no
se encontró ninguno. Y la lista de variables importantes que faltan, cuando se
pide explícitamente.

In [ ]:
from scoring_leads.columnas import ALIAS_COLUMNAS, VARIABLES_CLAVE, _clave, obtener_serie

print('Nombres distintos que colapsan a la misma clave comparable:')
for nombre in ['estado_civil', 'Estado Cívil', 'ESTADO CIVIL', 'estado civil', 'estado_civil_label']:
    print(f'   {nombre:<22} -> {_clave(nombre)}')
print()

print(f'{len(ALIAS_COLUMNAS)} variables reconocidas, {len(VARIABLES_CLAVE)} de ellas consideradas clave.')
print()

# Una exportación cruda del CRM: mismos datos, nombres crudos.
cruda = entrada.rename(columns={
    'sexo': 'sexo_label', 'estado_civil': 'estado_civil_label',
    'razon_de_compra': 'razoncompra_label', 'tipo_unidad': 'tipoapa_label',
    'ingreso_familiar': 'ingfam_label', 'canal_de_anuencia': 'anuencia_label',
    'canal_de_contacto': 'contacto_label', 'lugar_trabajo': 'lugtrab_label',
    'lugar_vivienda': 'lugvivi_label', 'opciones_visitadas': 'opciones_visitadas_label',
    'participantes_en_credito': 'partcredi_label',
})

print('Resolución de la variable estado_civil en cada versión de la tabla:')
print('   tabla canónica ->', obtener_serie(entrada, 'estado_civil').name)
print('   tabla cruda    ->', obtener_serie(cruda, 'estado_civil').name)
print()

sin_columna = entrada.drop(columns=['lugar_trabajo'])
serie = obtener_serie(sin_columna, 'lugar_trabajo')
print('Variable que no existe en la tabla: no falla, devuelve nulos.')
print('   largo:', len(serie), '| todos nulos:', serie.isna().all())
print()

print('variables_ausentes(tabla completa) :', variables_ausentes(entrada))
print('variables_ausentes(sin 3 columnas) :',
      variables_ausentes(entrada.drop(columns=['sexo', 'edad', 'ingreso_familiar'])))
print()

# La prueba de fondo: nombres crudos y canónicos deben dar exactamente lo mismo.
salida_cruda = agregar_cluster_y_percentil(cruda)
print('¿Misma tabla con nombres crudos produce el mismo resultado?')
print('   cluster  :', salida['cluster'].tolist() == salida_cruda['cluster'].tolist())
print('   percentil:', salida['percentil'].tolist() == salida_cruda['percentil'].tolist())

---

# 5. `scoring_leads/normalizacion.py` — limpieza de valores

### Propósito

Garantizar que un valor se limpie en producción **exactamente igual** que
durante el entrenamiento de los modelos. Esto es más importante de lo que
parece: si en la tesis el valor `'casado'` se convirtió en `'Casad@'`, y aquí
—por un descuido— se convirtiera en `'Casado'` (con mayúscula, sin la @), el
modelo vería una categoría que **nunca aprendió a reconocer** y perdería esa
información sobre el lead sin avisar a nadie. A ese problema —que la limpieza
de producción se desalinee poco a poco de la limpieza con la que se entrenó el
modelo— se le suele llamar *training-serving skew* en inglés; en español,
simplemente: **la limpieza de producción se separó de la del entrenamiento**.
Es una de las causas más comunes de que un modelo funcione bien en el notebook
de pruebas y falle, en silencio, una vez en producción.

La forma de evitarlo aquí es la más directa posible: **no volver a escribir la
lógica de limpieza**, sino **copiar literalmente** —línea por línea, sin
cambiar nada— las mismas funciones que se usaron para entrenar los modelos.

### Entradas

Valores sueltos, tal como vienen del CRM: texto con mayúsculas irregulares,
acentos ausentes, espacios de más, o marcadores de vacío escritos de formas
distintas (`'N/A'`, `'Sin dato'`, `'-'`).

### Qué hace

Es una copia **verbatim** (palabra que solo quiere decir «tal cual, sin cambiar
una letra») de las funciones de limpieza del pipeline de tesis. El encabezado
del archivo declara de dónde viene cada bloque:

| Función | Viene de | Qué resuelve |
|---|---|---|
| `normalize_for_comparison` | `cleaning_helpers.py` | Convierte un texto a una forma comparable: minúsculas, sin acentos, sin espacios de más |
| `clean_text_value` | `cleaning_helpers.py` | Pone cada palabra con su primera letra en mayúscula, pero respeta las siglas del negocio (`CRM`, `FHA`); los valores que en realidad significan «vacío» se convierten en nulo |
| `is_missing_or_not_registered` | `cleaning_helpers.py` | Decide si un valor debe tratarse como faltante |
| `normalize_sex` | `cleaning_helpers.py` | Homologa cómo está escrito el sexo (`'F'`, `'mujer'`, `'MASCULINO'`…) a una de las formas que el modelo reconoce; lo que no logra reconocer, lo marca como `'Otro'` |
| `normalize_marital_status` | `cleaning_helpers.py` | Homologa las variantes de soltero/casado a cuatro categorías fijas |
| `clean_channel_value_conservative` | `cleaning_helpers.py` | Corrige la escritura del canal de contacto, sin mezclar canales que en realidad son distintos |
| `classify_campania_tipo` / `_audiencia` | `feature_helpers.py` | Extraen el tipo de campaña y su público a partir del nombre de la campaña |

Dos matices que importan y que se conservaron tal cual:

* **`clean_channel_value_conservative` es conservadora a propósito.** Corrige
  mayúsculas, tildes y errores de escritura evidentes, pero **no junta**
  canales que la operación distingue: `Inbox` no se convierte en `WhatsApp`,
  `Fanpage Líderes` no se convierte en `Facebook`, `ExpoCasa` no se convierte en
  un `Evento` genérico. Un valor que no está en la lista de canales conocidos
  **se conserva tal cual** (con una limpieza mínima de formato), en vez de
  mandarlo a una categoría genérica `'Otro'` que borraría esa información.
* **`normalize_marital_status` nunca devuelve `'Otro'`.** Ante una variante que
  no había previsto, primero intenta reconocer fragmentos de palabra
  (`'casad'`, `'solter'`, `'conyuge'`) y, si tampoco encuentra nada, deja el
  valor limpio tal como venía, en vez de descartarlo.

Este archivo **no se edita solo**: si la limpieza necesita cambiar, el cambio
se hace primero en `01 Limpieza Código/scripts/` (donde vive la versión
original) y después se vuelve a copiar aquí, porque de lo contrario producción
y entrenamiento dejarían de coincidir — el problema que se explicó arriba.

> **Ejemplo real de ese procedimiento.** `normalize_sex` reconocía el sexo
> buscando fragmentos de palabra (`fem`, `mujer`, `masc`, `hombre`) dentro del
> valor. Una inicial suelta como `'F'` no contiene ninguno de esos fragmentos,
> así que caía en la categoría por defecto, `'Otro'` — una categoría que los
> modelos nunca vieron durante el entrenamiento. Se agregó el reconocimiento de
> los códigos de una sola letra (`'F'` y `'M'`), **primero** en
> `01 Limpieza Código/scripts/cleaning_helpers.py` y **después** copiado aquí,
> de forma que las dos versiones de la función siguen siendo idénticas. El
> cambio **no altera ningún resultado de la tesis**: se revisaron todos los
> valores de sexo que existen en los datos del proyecto y solo aparecen
> `'Femenino'` y `'Masculino'` — la regla nueva nunca llega a activarse sobre
> ellos —, y las comprobaciones de la sección 13 lo confirman con los
> resultados reales.

### Salidas

El valor ya limpio, o un valor nulo (`pd.NA`, la forma en que pandas representa
«este dato no existe») si el valor original en realidad significaba «vacío».

In [ ]:
from scoring_leads.normalizacion import (
    classify_campania_audiencia,
    classify_campania_tipo,
    clean_channel_value_conservative,
    clean_text_value,
    normalize_marital_status,
    normalize_sex,
)

def demo(titulo, funcion, valores):
    print(titulo)
    for valor in valores:
        resultado = funcion(valor)
        texto = 'NULO' if pd.isna(resultado) else repr(resultado)
        print(f'   {repr(valor):<28} -> {texto}')
    print()

demo('normalize_sex — homologa la escritura del sexo', normalize_sex,
     ['Femenino', 'F', 'MASCULINO', 'masc', 'mujer', 'Otro', None, 'Sin dato'])

demo('normalize_marital_status — cuatro categorías reales, nunca "Otro"', normalize_marital_status,
     ['Casad@', 'casado', 'CASADA', 'Solter@ + Hijos', 'Futuros Conyuges', 'X', None])

demo('clean_channel_value_conservative — corrige sin fusionar canales distintos',
     clean_channel_value_conservative,
     ['facebook', 'FB empresa_demo', 'whats app', 'Inbox', 'Fanpage Lideres',
      'pag web', 'Canal Nuevo 2026', 'N/A'])

demo('clean_text_value — title case, siglas intactas, vacíos disfrazados a nulo',
     clean_text_value,
     ['  zona   10 ', 'INVERSIÓN', 'crm interno', 'sin dato', '--', 3, None])

demo('classify_campania_tipo — lee el tipo en el nombre de la campaña', classify_campania_tipo,
     ['proyecto01 PRINCIPAL MAYO 25', 'proyecto05 MESSENGER', 'portal_demo EXPOCASA', 'proyecto04 LEADS EEUU LA',
      'Campaña Rara', None])

demo('classify_campania_audiencia — lee la audiencia geográfica', classify_campania_audiencia,
     ['proyecto04 LEADS EEUU LA', 'proyecto03 FOOTER USA CA', 'proyecto01 PRINCIPAL MAYO 25', None])

**Dos detalles de la salida anterior que conviene conocer**, porque afectan cómo
se lee el tablero:

* `normalize_sex` combina dos reglas: reconoce los **códigos de una sola letra**
  (`'F'`, `'M'`) y también los **fragmentos de palabra** (`fem`, `mujer`,
  `masc`, `hombre`). Lo que no encaja en ninguna de las dos —una `'X'`, un
  `'Indefinido'`— se homologa a `'Otro'`, que **no** es una de las dos
  categorías con las que se entrenó el clustering (`'Femenino'` /
  `'Masculino'`). Ese lead conserva su segmento, pero deja de aportar la
  variable sexo al cálculo (esto se explica con un ejemplo numérico en la
  sección 8). Es un argumento operativo para que el CRM capture el sexo con una
  lista desplegable de opciones fijas, en vez de dejar que se escriba como
  texto libre.
* `'Canal Nuevo 2026'` **sobrevive con su propio nombre**: un canal que la
  empresa estrene después de la tesis no se pierde ni se disfraza de otra cosa.
  Llega al modelo como una categoría que nunca vio antes, y ahí entra
  `OneHotEncoder(handle_unknown='ignore')` — la pieza de scikit-learn (ver
  Vocabulario: «codificación one-hot») configurada para que, ante una categoría
  desconocida, simplemente la trate como «ninguna de las conocidas» en vez de
  romperse con un error. Es la consecuencia práctica de que la limpieza sea
  conservadora.

---

# 6. `scoring_leads/preparacion.py` — construir lo que los modelos piden

### Propósito

Es el traductor entre «la tabla tal como existe» y «lo que los modelos
aprendieron a leer». Los dos modelos finales no reciben la tabla del CRM
directamente: esperan un conjunto exacto de columnas, con nombres exactos y
valores en el formato exacto del entrenamiento. Este módulo construye esa tabla
intermedia.

### Entradas

La tabla original (`DataFrame`), apoyándose en `columnas.py` para localizar
cada variable bajo cualquiera de sus alias, y en `normalizacion.py` para
limpiar sus valores.

### Qué hace

Tres bloques, en orden:

**1 · Limpieza.** Cada variable pasa por la función que le corresponde: `edad`
se convierte a número, `sexo` pasa por `normalize_sex`, los dos canales pasan
por `clean_channel_value_conservative`, y el resto de las columnas de texto
pasan por `clean_text_value`.

Aquí hay **una diferencia deliberada frente al entrenamiento**: una edad fuera
del rango 18–90 (como el `150` de la fila 104 del ejemplo) durante el
entrenamiento hacía que esa fila se eliminara del conjunto de datos; aquí, en
cambio, **el valor pasa a nulo y la fila sobrevive**. En producción no se puede
dar el lujo de perder un lead completo por un dato mal digitado — la etapa
de imputación del modelo (ver Vocabulario) se encarga de rellenar ese hueco.

**2 · Variables de campaña.** Si la tabla ya trae las columnas
`campania_tipo` y `campania_audiencia` ya calculadas, se reutilizan tal cual;
si no, se derivan a partir de `campania` con las mismas reglas del script
`04_build_final_features.py`. No se recalcula algo que ya venía resuelto.

**3 · Construcción de variables nuevas** (a veces se le llama «ingeniería de
variables»: el proceso de crear columnas nuevas a partir de las que ya
existen, porque le dan al modelo información que ninguna columna suelta
contiene). Diez de las 24 variables que necesita el modelo predictivo **no
existen en el CRM**: se calculan aquí, con las mismas reglas que se usaron en
la §21.4 del notebook final de la tesis.

| Variable nueva | Cómo se calcula | Qué información agrega |
|---|---|---|
| `ingreso_mid` | El punto medio del rango de ingreso que declaró el lead (diccionario `INCOME_MID`) | Convierte un rango de texto (`'Q30k A 33.9k'`) en un número que el modelo puede usar |
| `canal_familia` | Agrupa los ~27 canales de contacto en 7 familias más generales (diccionario `CANAL_FAMILIA`) | Reduce la cantidad de categorías distintas sin perder el sentido comercial de cada canal |
| `es_inversion` | `1` si `razon_de_compra` es `'Inversión'`, si no `0` | Es el corte de negocio más fuerte que se encontró en todo el proyecto |
| `es_referido` | `1` si el lead llegó por un canal de la familia «referido», si no `0` | Los leads referidos convierten distinto que el resto |
| `mayor_45`, `ingreso_alto` | `1` o `0` según si la edad o el ingreso pasan cierto umbral | Le permiten al modelo captar que el efecto de la edad o el ingreso no es necesariamente una línea recta: puede haber un salto a partir de cierto punto |
| `edad_x_inversion`, `edad_x_ingreso`, `edad_x_referido`, `ingreso_x_inversion` | El producto (la multiplicación) entre dos de las variables anteriores | Son **interacciones**: capturan que el efecto de la edad, por ejemplo, no es el mismo para un lead que invierte que para uno que busca vivienda — depende de la combinación de ambas variables, no de cada una por separado |

Un detalle final del módulo, pensado para producción: todas las columnas de
texto se fuerzan a quedar como texto (`str`), incluso si venían como número.
La razón: el modelo aprendió sus categorías leyendo un archivo CSV, donde todo
es texto — así que para el modelo, `participantes_en_credito` fue siempre el
texto `'2'`, nunca el número `2`. Si Power BI entrega esa misma columna como
número entero, y el código no hiciera esta conversión, el modelo vería
`2 != '2'` y trataría el valor como una categoría que nunca ha visto,
perdiendo esa información. Convertir todo a texto antes de llegar al modelo
evita ese problema.

### Salidas

Una tabla **nueva**, con las mismas filas en el mismo orden que la tabla de
entrada, y con las columnas que consumen los dos modelos. La tabla original
nunca se modifica.

In [ ]:
from scoring_leads.preparacion import (
    CANAL_FAMILIA, COLUMNAS_CATEGORICAS, EDAD_MIN, EDAD_MAX, INCOME_MID, preparar,
)

preparado = preparar(entrada)

print(f'Entrada : {entrada.shape[0]} filas x {entrada.shape[1]} columnas')
print(f'Preparado: {preparado.shape[0]} filas x {preparado.shape[1]} columnas')
print()
print('Columnas de la tabla de trabajo:')
DERIVADAS_DE_CAMPANIA = ('campania_tipo', 'campania_audiencia')
for i, columna in enumerate(preparado.columns, 1):
    if columna in DERIVADAS_DE_CAMPANIA:
        origen = 'DERIVADA de campania'
    elif columna in ALIAS_COLUMNAS:
        origen = 'del CRM (limpia)'
    else:
        origen = 'DERIVADA aquí (§21.4)'
    print(f'   {i:2}. {columna:<24} {origen}')

El efecto de la limpieza se ve mejor lado a lado. Nótese la fila 104
(`edad = 150`), que se convierte en nulo sin perder el lead, y la fila 105, donde
tres escrituras distintas se homologan a las categorías del entrenamiento.

In [ ]:
comparacion = pd.DataFrame({
    'id_trato'        : entrada['id_trato'].to_numpy(),
    'edad_cruda'      : entrada['edad'].to_numpy(),
    'edad_limpia'     : preparado['edad'].to_numpy(),
    'sexo_crudo'      : entrada['sexo'].to_numpy(),
    'sexo_limpio'     : preparado['sexo'].to_numpy(),
    'civil_crudo'     : entrada['estado_civil'].to_numpy(),
    'civil_limpio'    : preparado['estado_civil'].to_numpy(),
    'razon_cruda'     : entrada['razon_de_compra'].to_numpy(),
    'razon_limpia'    : preparado['razon_de_compra'].to_numpy(),
})
print(comparacion.to_string(index=False))
print()
print(f'Rango de edad válido: {EDAD_MIN}-{EDAD_MAX}. Fuera de rango -> nulo, la fila NO se elimina.')

In [ ]:
derivadas = ['ingreso_familiar', 'ingreso_mid', 'canal_de_anuencia', 'canal_familia',
             'es_inversion', 'es_referido', 'mayor_45', 'ingreso_alto',
             'edad_x_inversion', 'edad_x_ingreso', 'edad_x_referido', 'ingreso_x_inversion']

print('Las variables que el CRM no tiene y se construyen aquí:')
print(preparado[derivadas].to_string())
print()
print(f'INCOME_MID    : {len(INCOME_MID)} rangos de ingreso -> punto medio en miles de Q')
print(f'CANAL_FAMILIA : {len(CANAL_FAMILIA)} canales -> {len(set(CANAL_FAMILIA.values()))} familias '
      f'({", ".join(sorted(set(CANAL_FAMILIA.values())))})')
print()
print('Nota: "Rango Inventado" (fila 105) no está en INCOME_MID -> ingreso_mid queda nulo,')
print('y lo resuelve el imputador del pipeline. El lead no se pierde.')
print()
print('Tipo de las categóricas tras la homologación a texto:')
print('  ', {c: str(preparado[c].dtype) for c in list(COLUMNAS_CATEGORICAS)[:4]}, '...')
print('   participantes_en_credito, que llegó como entero:',
      [v for v in preparado['participantes_en_credito'].tolist()])

---

# 7. `scoring_leads/artefactos.py` — abrir los modelos, una sola vez

### Propósito

Centralizar la carga de los tres archivos de modelo, con dos requisitos: que las
rutas **nunca** sean absolutas, y que los archivos se lean **una sola vez** por
sesión.

### Entradas

Los tres archivos de `scoring_leads/modelos/`.

### Qué hace

Resuelve las rutas a partir de `Path(__file__).resolve().parent`, es decir la
ubicación del propio módulo. Una ruta absoluta escrita a mano
(`C:\Users\...\modelos\`) funcionaría en la máquina donde se escribió y fallaría
en cualquier otra; con rutas relativas la carpeta se puede mover, copiar a un
recurso de red o instalar con `pip`, y sigue encontrando sus modelos.

Cada función de carga lleva `@lru_cache(maxsize=1)`, un decorador de Python que
activa una **caché**: guarda en memoria el resultado de la primera vez que la
función se ejecuta, y en las llamadas siguientes devuelve ese mismo resultado
sin volver a hacer el trabajo. Power BI ejecuta el script una vez por
actualización, pero dentro de esa ejecución los artefactos se piden varias
veces (el `.json` lo consultan tanto `construir_matriz` como
`mascara_universo`). Sin caché, cada una de esas veces volvería a abrir el
archivo del disco; con ella, solo la primera lectura toca el disco y las demás
reutilizan lo que ya está en memoria.

`_verificar()` traduce el error críptico de un archivo ausente en un mensaje que
dice qué falta y qué hacer: copiar la carpeta `modelos` completa. Es el error más
frecuente al mover el paquete.

### Salidas

El modelo K-Means, el diccionario de preprocesamiento y el *bundle* del modelo
predictivo (pipeline + lista de variables).

In [ ]:
from scoring_leads.artefactos import (
    DIRECTORIO_MODELOS, cargar_kmeans, cargar_modelo_conversion, cargar_preprocesamiento_kmeans,
)

print('Directorio resuelto desde __file__ (nunca absoluto en el código):')
print('  ', DIRECTORIO_MODELOS)
print()

kmeans = cargar_kmeans()
prep = cargar_preprocesamiento_kmeans()
bundle = cargar_modelo_conversion()

print('1) modelo_kmeans_R1.joblib')
print(f'     tipo           : {type(kmeans).__name__}')
print(f'     k (clusters)   : {kmeans.n_clusters}')
print(f'     columnas       : {kmeans.n_features_in_}')
print(f'     centroides     : {kmeans.cluster_centers_.shape}')
print()

print('2) preprocesamiento_kmeans_R1.json  (parámetros congelados del entrenamiento)')
print(f'     entrenado con  : {prep["n_observaciones_entrenamiento"]:,} leads')
print(f'     edad media/desv: {prep["edad_scaler"]["mean"]:.4f} / {prep["edad_scaler"]["scale"]:.4f}')
print(f'     bloques        : {prep["orden_bloques"]}')
print(f'     pesos          : edad={prep["peso_edad"]}, {prep["pesos_categoricas"]}')
print()

print('3) modelo_conversion_lr.pkl')
print(f'     claves         : {list(bundle.keys())}')
print(f'     variables      : {len(bundle["features"])}')
print(f'     pasos          : {[nombre for nombre, _ in bundle["pipeline"].steps]}')
clasificador = bundle['pipeline'].steps[-1][1]
print(f'     clasificador   : {type(clasificador).__name__}(C={clasificador.C}, penalty={clasificador.penalty!r})')
print()

import time
inicio = time.perf_counter(); cargar_kmeans(); primera = time.perf_counter() - inicio
inicio = time.perf_counter(); cargar_kmeans(); segunda = time.perf_counter() - inicio
print(f'Efecto de lru_cache: 2a llamada {primera / max(segunda, 1e-9):,.0f}x más rápida (no vuelve al disco)')

---

# 8. `scoring_leads/clustering.py` — la columna `cluster`

### Propósito

Reproducir, para leads nuevos, la segmentación K-Means final de la tesis
(configuración llamada **R1**, con 6 grupos), respetando una condición que no
se puede negociar: **un mismo lead debe recibir siempre el mismo cluster**, sin
importar junto a qué otros leads se le esté puntuando en ese momento.

### Entradas

La tabla ya preparada (ver sección 6), el modelo entrenado
`modelo_kmeans_R1.joblib` y el archivo `preprocesamiento_kmeans_R1.json`, que
guarda los números exactos que se usaron para transformar los datos durante el
entrenamiento (más abajo se explica por qué eso es tan importante).

### Qué hace

Un modelo de K-Means no lee las columnas de un lead directamente: necesita que
cada lead se convierta primero en una fila de números — a esto se le llama la
**matriz** de entrada. Este módulo arma esa matriz de 13 números por lead,
exactamente igual que la función `construir_matriz_r1()` del notebook de la
tesis, pero con un cambio importante: usa los números **congelados** del
entrenamiento en vez de recalcularlos sobre los datos nuevos. Esta es la
decisión central del módulo, y vale la pena entender por qué.

Si la edad de cada lead se estandarizara (ver Vocabulario) usando el promedio
del lote de leads que se está puntuando *en ese momento*, entonces el mismo
lead podría terminar en un cluster distinto según con qué otros leads se le
compare — su segmento dejaría de ser una característica del lead y pasaría a
depender de quién más está en la tabla. Usando en cambio el promedio y la
variación que se calcularon **una sola vez**, durante el entrenamiento
(edad promedio 39.2034, con una variación típica de 10.9320), la asignación de
un lead a un cluster queda fija para siempre, sin importar en qué lote se le
puntúe.

La matriz tiene cinco bloques, uno por cada variable que usa el clustering.
Cada categoría de texto se convierte en varias columnas de 0 y 1 —la
**codificación one-hot** del Vocabulario— y luego se le aplica una división y
un peso:

| Bloque | Cuántas columnas ocupa | Cómo se transforma | Peso final |
|---|---|---|---|
| `edad` | 1 | Se estandariza con el promedio y la variación del entrenamiento | × 0.25 |
| `sexo` | 2 (una por categoría) | Codificación one-hot, dividida entre √2 | × 1.2 |
| `estado_civil` | 3 | Codificación one-hot, dividida entre √3 | × 1.5 |
| `razon_de_compra` | 2 | Codificación one-hot, dividida entre √2 | × 1.4 |
| `tipo_unidad` | 5 | Codificación one-hot, dividida entre √5 | × 1.0 |

¿Para qué dividir entre la **raíz cuadrada del número de categorías**? Sin esa
división, una variable con más categorías —como `tipo_unidad`, con 5— pesaría
automáticamente más en el cálculo de distancias que una con solo 2 categorías
—como `sexo`— simplemente por tener más columnas, no porque sea más
importante. Dividir entre esa raíz cuadrada equilibra el terreno antes de
aplicar los pesos que sí se eligieron a propósito (los de la última columna de
la tabla), que son los de la configuración R1 elegida en la Parte II de la
tesis.

**Datos faltantes y categorías nuevas** se resuelven con un valor «neutro»: una
edad vacía se estandariza como si fuera 0 (es decir, se trata como si tuviera
exactamente la edad promedio), y una categoría que el modelo nunca vio deja
todas las columnas de su bloque one-hot en cero — el mismo comportamiento que
tendría `OneHotEncoder(handle_unknown='ignore')`, la pieza de scikit-learn que
hace esto mismo automáticamente en otros módulos del proyecto.

**La función `mascara_universo()` y el valor −1.** El modelo se entrenó
únicamente con leads que tenían las **cinco** variables demográficas
completas. A un lead al que le falta alguna, esos valores «neutros» lo dejan
matemáticamente cerca del punto de origen (el centro del sistema de
coordenadas), donde los seis centroides quedan casi a la misma distancia entre
sí — el «más cercano» termina decidiéndolo el ruido numérico, no el verdadero
perfil del lead. Asignarle igual un cluster inundaría un grupo real con leads
que en realidad no se pudieron perfilar, y arruinaría la lectura de conversión
por cluster que se muestra en el tablero. Por eso, salvo que se pida
explícitamente lo contrario, esos leads reciben `-1`, que se lee como «sin
segmento».

**Un matiz de esa comprobación:** revisa que los cinco valores **no estén
vacíos**, no que pertenezcan a una de las categorías que el modelo vio durante
el entrenamiento. La fila 105 del ejemplo lo muestra: su `tipo_unidad` es
`'5 Habs'`, una categoría que el entrenamiento nunca tuvo. Ese lead sí cuenta
como parte del universo del modelo y recibe un segmento, pero su bloque de
tipo de unidad queda en ceros — así que se le perfila con cuatro variables que
sí aportan información, en vez de cinco. Se eligió este criterio a propósito:
una categoría poco común no debería costarle el segmento a un lead que sí
llenó los cinco campos, aunque uno de esos campos tenga un valor raro. Conviene
tenerlo presente al revisar el tablero.

### Salidas

Una lista de números enteros, uno por lead, en el mismo orden que la tabla de
entrada: valores del 0 al 5, o −1 para «sin segmento».

In [ ]:
from scoring_leads.clustering import (
    CLUSTER_SIN_SEGMENTO, asignar_cluster, construir_matriz, mascara_universo,
)

matriz = construir_matriz(preparado)
print(f'Matriz de clustering: {matriz.shape[0]} filas x {matriz.shape[1]} columnas '
      f'(el modelo espera {kmeans.n_features_in_})')
print()

etiquetas = []
for variable in prep['orden_bloques']:
    if variable == 'edad':
        etiquetas.append('edad(z)')
    else:
        etiquetas += [f'{variable[:9]}={c}' for c in prep['categorias'][variable]]

print('Composición columna por columna (fila 101, un lead completo):')
for etiqueta, valor in zip(etiquetas, matriz[0]):
    print(f'   {etiqueta:<26} {valor: .4f}')

In [ ]:
dentro = mascara_universo(preparado)
clusters = asignar_cluster(preparado)

resumen = pd.DataFrame({
    'id_trato'        : entrada['id_trato'].to_numpy(),
    'edad'            : preparado['edad'].to_numpy(),
    'sexo'            : preparado['sexo'].to_numpy(),
    'estado_civil'    : preparado['estado_civil'].to_numpy(),
    'razon_de_compra' : preparado['razon_de_compra'].to_numpy(),
    'tipo_unidad'     : preparado['tipo_unidad'].to_numpy(),
    '5 completas'     : dentro,
    'cluster'         : clusters,
})
print(resumen.to_string(index=False))
print()
print(f'Dentro del universo: {dentro.sum()} de {len(dentro)} leads')
print(f'Sin segmento (-1)  : {(clusters == CLUSTER_SIN_SEGMENTO).sum()} leads')

La justificación del −1 no es una opinión: se puede medir. Se toma un lead sin
ningún dato demográfico y se calculan sus distancias a los seis centroides.

In [ ]:
lead_vacio = pd.DataFrame({
    'edad': [np.nan], 'sexo': [None], 'estado_civil': [None],
    'razon_de_compra': [None], 'tipo_unidad': [None],
})
vector = construir_matriz(lead_vacio)
distancias = np.linalg.norm(kmeans.cluster_centers_ - vector, axis=1)

print('Lead sin ningún dato demográfico — distancia a cada centroide:')
for c, d in enumerate(distancias):
    marca = '  <- el "más cercano"' if d == distancias.min() else ''
    print(f'   cluster {c}: {d:.4f}{marca}')
print()
print(f'Rango de distancias : {distancias.min():.4f} a {distancias.max():.4f}')
print(f'Diferencia relativa : {(distancias.max() - distancias.min()) / distancias.min() * 100:.1f} % '
      f'entre el más cercano y el más lejano')
print()
print('Todos los centroides están casi a la misma distancia: la asignación la')
print('decidiría el ruido, no el perfil del lead. De ahí el -1.')
print()
print('Comportamiento por defecto        :', asignar_cluster(lead_vacio)[0])
print('Forzando asignar_fuera_de_universo:', asignar_cluster(lead_vacio, asignar_fuera_de_universo=True)[0])

---

# 9. `scoring_leads/prediccion.py` — la columna `percentil`

### Propósito

Calcular la probabilidad de conversión de cada lead con el modelo predictivo
final, y convertir ese número en la posición relativa que finalmente se
publica en el tablero.

### Entradas

La tabla ya preparada (sección 6) y el archivo `modelo_conversion_lr.pkl`, que
guarda el **pipeline** completo del modelo: la cadena de pasos que hay que
aplicarle a los datos, en orden, antes de obtener una probabilidad.

### Qué hace

Es el módulo más corto del paquete, y eso es intencional. El archivo del modelo
no guarda solo la fórmula final: guarda el **pipeline completo**, es decir,
los cuatro pasos encadenados que se le aplicaron a los datos durante el
entrenamiento:

1. **Imputación** (rellenar valores faltantes): `SimpleImputer`, la pieza de
   scikit-learn que rellena los huecos —con `'_missing_'` en las columnas de
   texto, con la mediana del entrenamiento en las columnas numéricas—.
2. **Codificación one-hot**: convierte cada categoría de texto en columnas de
   0 y 1, tal como se explicó en la sección 8.
3. **Estandarización**: pone las variables numéricas en una escala común.
4. **Regresión logística**: el modelo en sí, que con esos números ya
   preparados calcula la probabilidad de conversión.

Como esos cuatro pasos ya están guardados dentro del archivo, aquí **no se
reimplementa ninguno**: el código solo arma una tabla con las 24 variables que
el modelo necesita, en el orden exacto que el propio archivo declara, y le pide
al pipeline que calcule la probabilidad con `predict_proba` (el método de
scikit-learn que devuelve, para cada fila, la probabilidad de pertenecer a cada
categoría posible — aquí se usa la probabilidad de «sí convierte»). Copiar esos
pasos a mano, en vez de reutilizar el pipeline guardado, sería arriesgarse a
que en algún momento la copia y el original dejen de coincidir.

Los dos casos difíciles los resuelve el pipeline por sí solo, sin que este
módulo tenga que hacer nada extra:

* **valores faltantes** → los rellena el `SimpleImputer` mencionado arriba;
* **categorías nuevas que el modelo nunca vio** → las absorbe el
  `OneHotEncoder(handle_unknown='ignore')`, y las categorías que aparecían muy
  poco en el entrenamiento ya se habían agrupado con la opción
  `min_frequency=30` (es decir: toda categoría que aparecía menos de 30 veces
  se agrupó en un solo grupo «otras», en vez de dejarla suelta).

Si una variable entera no está en la tabla, se entrega vacía y el `SimpleImputer`
la resuelve igual: ningún lead se pierde por faltarle una columna completa.

**`calcular_percentil()`** convierte la probabilidad en un **ranking
porcentual** (ver Vocabulario): ordena a todos los leads de menor a mayor
probabilidad, y a cada uno le asigna el porcentaje de leads que quedaron por
debajo o igual a él. Cuando dos leads tienen exactamente la misma probabilidad,
reciben el promedio de las posiciones que ocuparían — así nunca queda uno
arbitrariamente por encima del otro solo por el orden en que aparecían en la
tabla.

**Por qué se publica el percentil y no la probabilidad directamente.** La tasa
de conversión general del negocio ronda el 3 %, así que las probabilidades
individuales son números pequeños (0.04, 0.07…) y fáciles de malinterpretar: si
se mostraran junto al nombre del cliente, invitarían a leerlas como una
certeza («este lead tiene 6 % de comprar»), que es precisamente lo que el
modelo no garantiza. Lo que el modelo sí sostiene, y sostiene bien, es el
**orden relativo**: en la validación de la tesis, si se llama primero al 10 %
de leads con mayor probabilidad, se encuentran 4.14 veces más conversiones que
llamando al azar. El percentil comunica exactamente eso —quién va primero en la
fila— sin prestarse a leerse como una promesa numérica.

**La contrapartida, que conviene tener presente:** el percentil es
**relativo al lote** que Power BI cargó en ese momento. No es una nota fija que
se pueda comparar entre un informe de hoy y uno de la semana pasada si los
lotes de leads cambiaron.

### Salidas

Una columna interna de probabilidades (que nunca se publica) y la columna
`percentil`, con valores de 0 a 100.

In [ ]:
from scoring_leads.prediccion import calcular_percentil, probabilidad_conversion, variables_requeridas

requeridas = variables_requeridas()
numericas = [v for v in requeridas if v in preparado.columns and pd.api.types.is_numeric_dtype(preparado[v])]
categoricas = [v for v in requeridas if v not in numericas]

print(f'El modelo declara {len(requeridas)} variables de entrada:')
print(f'   numéricas   ({len(numericas):2}): {", ".join(numericas)}')
print(f'   categóricas ({len(categoricas):2}): {", ".join(categoricas)}')
print()
print('¿Están todas en la tabla preparada?',
      all(v in preparado.columns for v in requeridas))
print()

probabilidad = probabilidad_conversion(preparado)
percentil = calcular_percentil(probabilidad)

tabla = pd.DataFrame({
    'id_trato'    : entrada['id_trato'].to_numpy(),
    'probabilidad': probabilidad.to_numpy().round(6),
    'percentil'   : percentil.to_numpy(),
}).sort_values('probabilidad', ascending=False)
print('Probabilidad (interna) y percentil (publicado), ordenados de mayor a menor:')
print(tabla.to_string(index=False))
print()
print('Mayor probabilidad implica mayor percentil:',
      bool(np.all(np.diff(tabla['percentil'].to_numpy()) <= 0)))
print('La probabilidad NO aparece en la salida final:',
      'probabilidad' not in salida.columns)

La naturaleza relativa del percentil se demuestra puntuando el mismo lead en dos
lotes distintos: su probabilidad no se mueve ni un decimal —depende solo de sus
datos—, pero su percentil sí, porque cambia contra quién se le compara. Es el
comportamiento correcto, y la razón por la que la sección 6.3 del Paso_2_Guia_Scoring advierte
que filtrar en el tablero no recalcula la columna.

In [ ]:
lote_completo = agregar_cluster_y_percentil(entrada)
lote_parcial  = agregar_cluster_y_percentil(entrada.iloc[:3])

comparativa = pd.DataFrame({
    'id_trato'            : lote_parcial['id_trato'].to_numpy(),
    'percentil en 7 leads': lote_completo['percentil'].to_numpy()[:3],
    'percentil en 3 leads': lote_parcial['percentil'].to_numpy(),
})
print(comparativa.to_string(index=False))
print()
prob_completo = probabilidad_conversion(preparar(entrada)).to_numpy()[:3]
prob_parcial  = probabilidad_conversion(preparar(entrada.iloc[:3])).to_numpy()
print('La probabilidad subyacente es idéntica en ambos lotes:',
      bool(np.allclose(prob_completo, prob_parcial)))

---

# 10. `scoring_leads/modelos/` — los tres artefactos

### Propósito

Guardar lo que los modelos aprendieron. Son datos, no código: no se abren ni se
editan a mano.

| Archivo | Qué guarda | De dónde viene |
|---|---|---|
| `modelo_kmeans_R1.joblib` | El K-Means entrenado: 6 centroides de 13 dimensiones | `02 K Means/Output/modelo_final_notebook/modelo_KMeans_R1.joblib` |
| `preprocesamiento_kmeans_R1.json` | Media y desviación de la edad, categorías vistas y pesos de la configuración R1 | Generado al congelar `construir_matriz_r1()` del notebook final |
| `modelo_conversion_lr.pkl` | El pipeline completo del modelo predictivo + la lista de sus 24 variables | `03 Prediccion/qualified_run/models/modelo_final_busqueda.pkl` |



In [ ]:
import hashlib, json

print(json.dumps(prep, indent=2, ensure_ascii=False)[:1400])
print('...')
print()

def huella(ruta):
    return hashlib.md5(Path(ruta).read_bytes()).hexdigest()

print('Huella MD5 de cada artefacto (sirve para confirmar que no se alteraron):')
for nombre in artefactos:
    ruta = DIRECTORIO_MODELOS / nombre
    print(f'   {nombre:<34} {huella(ruta)}')

---

# 11. `pruebas/test_contrato_dataset.py` 

### Propósito

Convertir en código ejecutable las promesas que el Paso_2_Guia_Scoring le hace al usuario. Un
documento que dice «no se pierden filas» es una intención; una prueba que lo
comprueba en cada ejecución es una garantía.

### Entradas

La tabla de ejemplo de la sección 2, con sus casos difíciles deliberados.

### Qué hace

Doce pruebas, agrupadas en tres familias:

**Integridad de la tabla** — que no cambie el número de filas ni el orden, que las
columnas nuevas sean exactamente dos, que las originales queden idénticas, y que
el `DataFrame` de entrada no se mute.

**Corrección de los valores** — que todo cluster esté en 0–5 o sea −1, que solo
los leads con las cinco variables completas reciban segmento, que un lead sin
datos no caiga en un cluster real, que el percentil esté en 0–100 y sea monótono
respecto de la probabilidad, y que **ninguna columna delate la probabilidad**.

**Robustez** — tabla vacía, tabla a la que le faltan columnas, y tabla con los
nombres crudos del CRM (`*_label`), que debe producir un resultado idéntico al de
la tabla canónica.

Se ejecuta con `pytest` (una herramienta estándar de Python para correr pruebas)
o directamente con `python`, sin dependencias extra: en la máquina donde vive
Power BI no siempre hay un entorno de desarrollo instalado.

### Salidas

Una línea por prueba y un recuento final.

In [ ]:
import subprocess

resultado = subprocess.run(
    [sys.executable, str(CARPETA / 'pruebas' / 'test_contrato_dataset.py')],
    capture_output=True, text=True, cwd=str(CARPETA),
)
print(resultado.stdout)
if resultado.returncode != 0:
    print('STDERR:', resultado.stderr[-2000:])
print('Código de salida:', resultado.returncode, '(0 = todas superadas)')

---

# 12. `requirements.txt` y `pyproject.toml` — el entorno

### Propósito

Dejar el entorno reproducible. 
### Qué hacen

**`requirements.txt`** lista las bibliotecas necesarias. La línea decisiva es
`scikit-learn==1.9.0`. Un `.pkl` es una fotografía de
objetos de una versión concreta de la biblioteca; con otra versión mayor puede
fallar al abrirse o, peor, abrirse y comportarse distinto. También incluye
`matplotlib`, que el scoring no usa: Power BI lo exige para habilitar los pasos de
Python en Power Query.

**`pyproject.toml`** convierte la carpeta en un paquete instalable con
`pip install .`. La sección `[tool.setuptools.package-data]` es la que evita el
error más común al distribuirlo: sin ella, `pip` copiaría los `.py` pero **dejaría
fuera los modelos** —`.joblib`, `.pkl`, `.json` no son código—, y el paquete
instalado fallaría al primer uso con un `FileNotFoundError`.

### Salidas

Un entorno donde `import scoring_leads` funciona y encuentra sus artefactos.

In [ ]:
for archivo in ('requirements.txt', 'pyproject.toml'):
    print('=' * 78)
    print(archivo)
    print('=' * 78)
    print((CARPETA / archivo).read_text(encoding='utf-8'))

---

# 14. Resumen y guía de mantenimiento

## Los componentes de un vistazo

| # | Archivo | Responsabilidad | Recibe | Entrega |
|---|---|---|---|---|
| 1 | `powerbi_script.py` | Lo que se pega en Power BI | `dataset` | `dataset` con 2 columnas |
| 2 | `scoring_leads/__init__.py` | Orquesta y verifica el contrato | La tabla de Power BI | La tabla + `cluster` + `percentil` |
| 3 | `columnas.py` | Reconoce los nombres del CRM | Nombres de columna | La `Series` correcta, o nulos |
| 4 | `normalizacion.py` | Limpia valores (copia del pipeline) | Valores crudos | Valores homologados |
| 5 | `preparacion.py` | Construye las variables de los modelos | Tabla cruda | Tabla de trabajo limpia y derivada |
| 6 | `artefactos.py` | Carga los modelos una sola vez | Rutas relativas | Modelos en memoria |
| 7 | `clustering.py` | Calcula `cluster` | Tabla preparada | Enteros −1 … 5 |
| 8 | `prediccion.py` | Calcula `percentil` | Tabla preparada | Decimales 0 … 100 |
| 9 | `modelos/` | Lo aprendido en la tesis | — | 3 artefactos |
| 10 | `pruebas/` | Verifica las 12 garantías | Tabla de ejemplo | `12/12 pruebas superadas` |

## Qué tocar cuando algo cambie

| Si cambia… | Se toca | Y después |
|---|---|---|
| El CRM renombra una columna | `columnas.py`, diccionario `ALIAS_COLUMNAS` | Ejecutar las pruebas |
| Aparece un canal nuevo | `preparacion.py`, `CANAL_FAMILIA` (si no está, cae en `'otro'` y no rompe nada) | Ejecutar las pruebas |
| Cambian los rangos de ingreso | `preparacion.py`, `INCOME_MID` | Ejecutar las pruebas |
| Se reentrena el K-Means | Sustituir `modelo_kmeans_R1.joblib` **y** `preprocesamiento_kmeans_R1.json` juntos | Rehacer la sección 13 |
| Se reentrena el modelo predictivo | Sustituir `modelo_conversion_lr.pkl` | Rehacer la sección 13 |
| Cambia la limpieza en el pipeline de tesis | **Primero** `01 Limpieza Código/scripts/`, luego copiar a `normalizacion.py` | Reentrenar y rehacer la sección 13 |

> Los dos artefactos del clustering **viajan siempre juntos**: el `.json` guarda
> la media, la desviación y las categorías con las que se construyó la matriz que
> ese `.joblib` aprendió. Sustituir uno solo produce un modelo que corre sin
> error y devuelve segmentos equivocados — el peor de los fallos, porque es
> silencioso.

## Las reglas que este paquete no rompe

1. **La tabla que entra es la tabla que sale**, más dos columnas. Nunca menos
   filas, nunca otro orden, nunca una columna original modificada.
2. **El mismo lead recibe siempre el mismo `cluster`**, porque el preprocesamiento
   está congelado y no se reajusta al lote.
3. **La probabilidad no se publica.** Se calcula, ordena y se descarta.
4. **Ningún lead se pierde** por un dato faltante, una categoría nueva o una
   columna ausente.
5. **La limpieza de producción es la misma que la de entrenamiento**, por copia
   literal y no por reimplementación.

---

*Fin del apéndice. Para el uso operativo del tablero, ver
[`Paso_2_Guia_Scoring.pdf`](Paso_2_Guia_Scoring.pdf).*